# Advanced: Multi-Strategy Field Removal with PAMOLA.CORE

**Goal**: Master all 3 field removal strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Remove explicit field list (sensitive PII)
- **Strategy 2**: Remove by pattern matching (regex-based)
- **Strategy 3**: Remove by characteristics (null %, low cardinality)
- Calculate advanced privacy metrics
- Analyze memory reduction
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/remove_fields/
        └── 02_remove_fields_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.field_ops.remove_fields import RemoveFieldsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record customer dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 15 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Dataset features:**
- 1000 customer records
- Multiple field types (PII, metadata, transactional)
- Pattern-based field naming (prefix_*, *_id)
- Fields with varying null percentages

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'remove_fields_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate names
    first_names = ['John', 'Jane', 'Bob', 'Alice', 'Charlie', 'Diana', 'Eve', 'Frank',
                   'Grace', 'Henry', 'Ivy', 'Jack', 'Kelly', 'Liam', 'Mia', 'Noah',
                   'Olivia', 'Paul', 'Quinn', 'Ryan']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller',
                  'Davis', 'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez',
                  'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin']
    
    # Generate cities and states
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix',
              'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose']
    states = ['NY', 'CA', 'IL', 'TX', 'AZ', 'PA', 'CA', 'CA', 'TX', 'CA']
    
    # Create city-state mapping
    city_state_map = dict(zip(cities, states))
    
    # Generate city data first, then map to states
    city_data = np.random.choice(cities, n_records)
    state_data = [city_state_map[city] for city in city_data]
    
    df = pd.DataFrame({
        'customer_id': range(1, n_records + 1),
        
        # PII fields (sensitive - for Strategy 1)
        'first_name': np.random.choice(first_names, n_records),
        'last_name': np.random.choice(last_names, n_records),
        'email': [f"user{i}@email.com" for i in range(1, n_records + 1)],
        'phone': [f"555-{str(i).zfill(4)}" for i in range(1, n_records + 1)],
        'ssn': [f"{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}" for _ in range(n_records)],
        
        # Pattern-based fields (for Strategy 2)
        'internal_customer_code': [f"CUST{str(i).zfill(6)}" for i in range(1, n_records + 1)],
        'internal_transaction_id': [f"TXN{str(i).zfill(8)}" for i in range(1, n_records + 1)],
        'internal_notes': [f"Internal note {i}" for i in range(1, n_records + 1)],
        
        # Fields with high null percentage (for Strategy 3) - FIX: Use None instead of np.nan
        'optional_field1': [None if np.random.random() > 0.1 else f"Value{i}" for i in range(n_records)],
        'optional_field2': [None if np.random.random() > 0.15 else f"Value{i}" for i in range(n_records)],
        
        # Low cardinality fields (for Strategy 3)
        'status': np.random.choice(['Active', 'Inactive'], n_records),
        'tier': np.random.choice(['Gold', 'Silver'], n_records),
        
        # Keep fields (important for analysis)
        'city': city_data,
        'state': state_data,
        'purchase_amount': np.random.uniform(10, 5000, n_records).round(2),
        'purchase_date': pd.date_range('2024-01-01', periods=n_records, freq='h').strftime('%Y-%m-%d'),
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    null_pct = df[col].isna().mean() * 100
    
    if pd.api.types.is_numeric_dtype(df[col]):
        # Numeric columns: show range
        print(f"  {col:30s} ({dtype_str:10s}): range=[{df[col].min():.2f}, {df[col].max():.2f}], {null_pct:5.1f}% null")
    else:
        # String/Object columns: show unique count
        print(f"  {col:30s} ({dtype_str:10s}): {df[col].nunique():4d} unique, {null_pct:5.1f}% null")

print(f"\n💡 Perfect dataset for testing all 3 removal strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for each strategy
   - `strategy1_fields`: List of sensitive PII fields
   - `strategy2_pattern`: Regex pattern for internal fields
   - `strategy3_null_threshold`: Null percentage threshold (0.0-1.0)
   - `strategy3_cardinality_threshold`: Max unique values
2. Run to validate configuration and setup environment

**What you'll see:**
- Strategy 1: Field list validation with unique counts
- Strategy 2: Pattern preview showing matched fields
- Strategy 3: Threshold validation and affected fields preview
- Environment confirmations (✓ for each component)

**Note:** All fields will be validated before execution to prevent errors

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    # Strategy 1: Explicit list of sensitive PII fields
    "strategy1_fields": ["email", "phone", "ssn"],
    
    # Strategy 2: Pattern to match internal/system fields
    "strategy2_pattern": r"^internal_.*",  # Matches fields starting with 'internal_'
    
    # Strategy 3: Characteristics thresholds
    "strategy3_null_threshold": 0.80,  # Remove fields with >80% null values
    "strategy3_cardinality_threshold": 3,  # Remove fields with ≤3 unique values
}

# Validate Strategy 1 fields
print("📋 Validating Strategy 1: Explicit Field List...\n")
for field_name in FIELD_CONFIG["strategy1_fields"]:
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG['strategy1_fields']"
        )
    print(f"  ✓ '{field_name}': {df[field_name].nunique()} unique values")

# Validate Strategy 2 pattern
print(f"\n📋 Validating Strategy 2: Pattern Matching...\n")
print(f"  Pattern: {FIELD_CONFIG['strategy2_pattern']}")
import re
pattern_matches = [col for col in df.columns if re.search(FIELD_CONFIG['strategy2_pattern'], col)]
if not pattern_matches:
    print(f"  ⚠️  Warning: Pattern matches no fields!")
else:
    print(f"  ✓ Matches {len(pattern_matches)} fields: {', '.join(pattern_matches)}")

# Validate Strategy 3 thresholds
print(f"\n📋 Validating Strategy 3: Characteristics-Based...\n")
print(f"  Null threshold: >{FIELD_CONFIG['strategy3_null_threshold']*100:.0f}%")
print(f"  Cardinality threshold: ≤{FIELD_CONFIG['strategy3_cardinality_threshold']} unique values")

# Preview fields that would be removed
high_null_fields = [col for col in df.columns if df[col].isna().mean() > FIELD_CONFIG['strategy3_null_threshold']]
low_card_fields = [col for col in df.columns if df[col].nunique() <= FIELD_CONFIG['strategy3_cardinality_threshold']]
combined_fields = list(set(high_null_fields + low_card_fields))

if high_null_fields:
    print(f"  ✓ High null fields: {', '.join(high_null_fields)}")
if low_card_fields:
    print(f"  ✓ Low cardinality fields: {', '.join(low_card_fields)}")
if combined_fields:
    print(f"  Total fields to remove: {len(combined_fields)}")
else:
    print(f"  ⚠️  No fields meet removal criteria")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_removal",
    description="Multi-strategy field removal processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Remove Explicit Field List

**How to use:**
- Removes specific sensitive PII fields by name
- Uses explicit list from FIELD_CONFIG
- Run to execute strategy

**Key parameters:**
- `fields_to_remove`: List of field names (email, phone, ssn)
- `pattern=None`: No pattern matching
- `output_format='csv'`: Save as CSV

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time (1-3 seconds expected)
- Field reduction (17 → 14 fields)

**Note:** Best for removing known sensitive fields with explicit control

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: REMOVE EXPLICIT FIELD LIST")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Explicit list",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = RemoveFieldsOperation(
    # Core parameters
    fields_to_remove=FIELD_CONFIG["strategy1_fields"],
    pattern=None,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Fields to remove: {FIELD_CONFIG['strategy1_fields']}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_explicit',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_explicit' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    print(f"\n📊 Reduction: {len(df.columns)} → {len(result_df_s1.columns)} fields")
    print(f"   Fields removed: {len(df.columns) - len(result_df_s1.columns)}")

## STRATEGY 2: Remove by Pattern Matching

**How to use:**
- Removes fields matching regex pattern
- Targets internal/system fields automatically
- Run to execute strategy

**Key parameters:**
- `fields_to_remove=None`: No explicit list
- `pattern=r"^internal_.*"`: Regex for fields starting with 'internal_'
- `output_format='csv'`: Save as CSV

**What you'll see:**
- Configuration summary with pattern
- Progress: validation → pattern matching → processing → metrics → viz → save
- Completion time (1-3 seconds expected)
- Field reduction showing pattern-matched removals

**Note:** Best for removing fields by naming convention without manual listing

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: REMOVE BY PATTERN MATCHING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Pattern",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = RemoveFieldsOperation(
    # Core parameters
    fields_to_remove=None,
    pattern=FIELD_CONFIG["strategy2_pattern"],
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Pattern: {FIELD_CONFIG['strategy2_pattern']}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_pattern',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_pattern' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    print(f"\n📊 Reduction: {len(result_df_s1.columns)} → {len(result_df_s2.columns)} fields")
    print(f"   Fields removed: {len(result_df_s1.columns) - len(result_df_s2.columns)}")

## STRATEGY 3: Remove by Characteristics

**How to use:**
- Removes fields based on data quality metrics
- Targets high-null and low-cardinality fields
- Run to execute strategy

**Implementation:**
This strategy requires custom logic to identify fields meeting criteria:
- Calculate null percentage per field
- Calculate unique value count per field
- Build removal list from thresholds

**What you'll see:**
- Field analysis (null %, cardinality)
- Fields meeting removal criteria
- Progress: analysis → processing → metrics → viz → save
- Completion time (1-3 seconds expected)
- Field reduction based on data quality

**Note:** Best for automated data quality-driven field reduction

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: REMOVE BY CHARACTERISTICS")
print("=" * 80 + "\n")

# Analyze fields to determine which to remove
print("📊 Analyzing field characteristics...\n")

fields_to_remove_s3 = []

# Find high-null fields
null_threshold = FIELD_CONFIG["strategy3_null_threshold"]
high_null_fields = [
    col for col in result_df_s2.columns 
    if result_df_s2[col].isna().mean() > null_threshold
]
if high_null_fields:
    print(f"  High null fields (>{null_threshold*100:.0f}%):")
    for field in high_null_fields:
        null_pct = result_df_s2[field].isna().mean() * 100
        print(f"    - {field}: {null_pct:.1f}% null")
    fields_to_remove_s3.extend(high_null_fields)

# Find low-cardinality fields
card_threshold = FIELD_CONFIG["strategy3_cardinality_threshold"]
low_card_fields = [
    col for col in result_df_s2.columns
    if result_df_s2[col].nunique() <= card_threshold
]
if low_card_fields:
    print(f"\n  Low cardinality fields (≤{card_threshold} unique):")
    for field in low_card_fields:
        unique_count = result_df_s2[field].nunique()
        print(f"    - {field}: {unique_count} unique values")
    fields_to_remove_s3.extend(low_card_fields)

# Remove duplicates
fields_to_remove_s3 = list(set(fields_to_remove_s3))

if not fields_to_remove_s3:
    print("\n⚠️  No fields meet removal criteria")
    print("   Skipping Strategy 3")
    result_df_s3 = result_df_s2
    elapsed_s3 = 0
else:
    print(f"\n  Total fields to remove: {len(fields_to_remove_s3)}")
    print(f"  Fields: {', '.join(fields_to_remove_s3)}")
    
    tracker_s3 = HierarchicalProgressTracker(
        total=6,
        description="Strategy 3: Characteristics",
        unit="steps",
        track_memory=True,
        level=0
    )
    
    operation_s3 = RemoveFieldsOperation(
        fields_to_remove=fields_to_remove_s3,
        pattern=None,
        output_format='csv',
        use_vectorization=False,
        parallel_processes=1,
        use_cache=False,
        generate_visualization=True,
        save_output=True,
        force_recalculation=False
    )
    
    print("\n✓ Strategy 3 configured")
    print(f"\n⚙️  Executing...\n")
    start_time = time.time()
    
    # Use result from Strategy 2 as input
    data_source_s3 = DataSource(dataframes={"main_dataset": result_df_s2})
    
    result_s3 = operation_s3.execute(
        data_source=data_source_s3,
        task_dir=task_dir / 'strategy3_characteristics',
        reporter=task_reporter,
        progress_tracker=tracker_s3,
        **kwargs
    )
    
    elapsed_s3 = time.time() - start_time
    print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")
    
    # Load results
    output_files_s3 = sorted(
        list((task_dir / 'strategy3_characteristics' / 'output').glob('*.csv')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    if output_files_s3:
        result_df_s3 = pd.read_csv(output_files_s3[0])
        print(f"\n📊 Reduction: {len(result_df_s2.columns)} → {len(result_df_s3.columns)} fields")
        print(f"   Fields removed: {len(result_df_s2.columns) - len(result_df_s3.columns)}")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times and field reduction
- Compare effectiveness across strategies
- Identify best approach for your use case

**What you'll see:**
- Execution time for each strategy
- Progressive field reduction (17 → 14 → 11 → 9 fields)
- Total processing time
- Memory reduction per strategy

**Note:** Combined strategies provide maximum field reduction with minimal execution time

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Explicit):        {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Pattern):         {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Characteristics): {elapsed_s3:6.2f}s")
print(f"  Total:                        {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Field Reduction:")
print(f"  Original:          {len(df.columns):3d} fields")
print(f"  After Strategy 1:  {len(result_df_s1.columns):3d} fields ({len(df.columns) - len(result_df_s1.columns)} removed)")
print(f"  After Strategy 2:  {len(result_df_s2.columns):3d} fields ({len(result_df_s1.columns) - len(result_df_s2.columns)} removed)")
if 'result_df_s3' in locals():
    print(f"  After Strategy 3:  {len(result_df_s3.columns):3d} fields ({len(result_df_s2.columns) - len(result_df_s3.columns)} removed)")
    print(f"  Total removed:     {len(df.columns) - len(result_df_s3.columns):3d} fields ({(len(df.columns) - len(result_df_s3.columns)) / len(df.columns) * 100:.1f}%)")

# Memory comparison
print(f"\n💾 Memory Reduction:")
mem_original = df.memory_usage(deep=True).sum() / 1024
mem_s1 = result_df_s1.memory_usage(deep=True).sum() / 1024
mem_s2 = result_df_s2.memory_usage(deep=True).sum() / 1024
if 'result_df_s3' in locals():
    mem_s3 = result_df_s3.memory_usage(deep=True).sum() / 1024
    print(f"  Original:          {mem_original:8.2f} KB")
    print(f"  After Strategy 1:  {mem_s1:8.2f} KB ({(mem_original - mem_s1) / mem_original * 100:5.1f}% reduction)")
    print(f"  After Strategy 2:  {mem_s2:8.2f} KB ({(mem_original - mem_s2) / mem_original * 100:5.1f}% reduction)")
    print(f"  After Strategy 3:  {mem_s3:8.2f} KB ({(mem_original - mem_s3) / mem_original * 100:5.1f}% reduction)")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Field counts, memory usage, execution time
2. **Visualizations**: Field/memory comparisons (first 2 displayed)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_explicit', 'Strategy 1: Explicit List'),
    ('strategy2_pattern', 'Strategy 2: Pattern Matching'),
    ('strategy3_characteristics', 'Strategy 3: Characteristics')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                print(f"   Fields removed: {metrics.get('fields_removed_count', 0)}")
                print(f"   Reduction: {metrics.get('fields_removed_percentage', 0):.2f}%")
                
                mem = metrics.get('memory_usage_byte', {})
                if isinstance(mem, dict):
                    before = mem.get('before', 0)
                    after = mem.get('after', 0)
                    if before > 0:
                        reduction = (before - after) / before * 100
                        print(f"   Memory reduction: {reduction:.2f}%")
                
                print(f"   Execution: {metrics.get('execution_time_seconds', 0):.4f}s")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate field reduction and memory savings
- Compare privacy improvements across strategies
- Run AFTER all strategies complete

**What you'll see:**
- Original field count and memory
- Progressive reduction per strategy
- Final metrics (field count, memory, reduction %)
- Summary of removed field types

**Privacy benefits:**
- Reduced attack surface (fewer sensitive fields)
- Lower storage requirements
- Improved data minimization compliance

**Note:** Each strategy contributes to overall privacy improvement

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

# Field reduction metrics
print("📊 FIELD REDUCTION:")
print(f"  Original fields:        {len(df.columns)}")
if 'result_df_s3' in locals():
    print(f"  Final fields:           {len(result_df_s3.columns)}")
    print(f"  Total removed:          {len(df.columns) - len(result_df_s3.columns)} ({(len(df.columns) - len(result_df_s3.columns)) / len(df.columns) * 100:.1f}%)")
    
    # Categorize removed fields
    removed_fields = [col for col in df.columns if col not in result_df_s3.columns]
    pii_fields = [f for f in removed_fields if f in FIELD_CONFIG['strategy1_fields']]
    internal_fields = [f for f in removed_fields if re.search(FIELD_CONFIG['strategy2_pattern'], f)]
    quality_fields = [f for f in removed_fields if f not in pii_fields and f not in internal_fields]
    
    print(f"\n📋 REMOVED FIELD BREAKDOWN:")
    if pii_fields:
        print(f"  Sensitive PII:          {len(pii_fields)} fields ({', '.join(pii_fields)})")
    if internal_fields:
        print(f"  Internal/System:        {len(internal_fields)} fields ({', '.join(internal_fields)})")
    if quality_fields:
        print(f"  Low quality:            {len(quality_fields)} fields ({', '.join(quality_fields)})")

# Memory reduction metrics
print(f"\n💾 MEMORY REDUCTION:")
original_memory = df.memory_usage(deep=True).sum()
if 'result_df_s3' in locals():
    final_memory = result_df_s3.memory_usage(deep=True).sum()
    memory_reduction = (original_memory - final_memory) / original_memory * 100
    
    print(f"  Original memory:        {original_memory / 1024:.2f} KB")
    print(f"  Final memory:           {final_memory / 1024:.2f} KB")
    print(f"  Reduction:              {memory_reduction:.1f}%")
    print(f"  Saved:                  {(original_memory - final_memory) / 1024:.2f} KB")

# Privacy impact
print(f"\n✨ PRIVACY IMPACT:")
if 'result_df_s3' in locals():
    print(f"  ✓ Sensitive PII fields removed")
    print(f"  ✓ Internal metadata stripped")
    print(f"  ✓ Low-quality fields cleaned")
    print(f"  ✓ Attack surface reduced by {(len(df.columns) - len(result_df_s3.columns)) / len(df.columns) * 100:.1f}%")
    print(f"  ✓ Storage requirements reduced by {memory_reduction:.1f}%")

## Step 7: Export Final Data

**How to use:**
- Export final datasets from each strategy
- Each strategy saves to its own folder
- Run AFTER all strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (path, rows, columns)
- Preview of first 5 rows
- Field count summary

**Output structure:**
```
advanced_output/
├── strategy1_explicit/removed_fields_data.csv
├── strategy2_pattern/removed_fields_data.csv
└── strategy3_characteristics/removed_fields_data.csv
```

**Note:** Progressive field reduction visible across strategy outputs

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Explicit Field List
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: EXPLICIT FIELD LIST")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_explicit'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s1 = s1_dir / 'removed_fields_data.csv'
    try:
        result_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(result_df_s1):,}")
        print(f"   Columns: {len(result_df_s1.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s1.head())
    
    print(f"\n📈 Fields: {len(result_df_s1.columns)}")
    print(f"   {', '.join(result_df_s1.columns[:10])}{'...' if len(result_df_s1.columns) > 10 else ''}")

# ============================================================================
# STRATEGY 2: Pattern Matching
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: PATTERN MATCHING")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_pattern'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s2 = s2_dir / 'removed_fields_data.csv'
    try:
        result_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(result_df_s2):,}")
        print(f"   Columns: {len(result_df_s2.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s2.head())
    
    print(f"\n📈 Fields: {len(result_df_s2.columns)}")
    print(f"   {', '.join(result_df_s2.columns[:10])}{'...' if len(result_df_s2.columns) > 10 else ''}")

# ============================================================================
# STRATEGY 3: Characteristics
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: CHARACTERISTICS")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_characteristics'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s3 = s3_dir / 'removed_fields_data.csv'
    try:
        result_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(result_df_s3):,}")
        print(f"   Columns: {len(result_df_s3.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    print(result_df_s3.head())
    
    print(f"\n📈 Fields: {len(result_df_s3.columns)}")
    print(f"   {', '.join(result_df_s3.columns)}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 field removal strategies implemented and compared
- ✅ Privacy metrics calculated (field/memory reduction)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Strategy 1 (Explicit): Best for known sensitive fields with precise control
- Strategy 2 (Pattern): Best for removing fields by naming convention
- Strategy 3 (Characteristics): Best for automated data quality cleanup
- Combined approach provides maximum privacy with minimal overhead

**Strategy recommendations:**
- **Use Strategy 1** when: You have specific PII fields to remove (GDPR, CCPA compliance)
- **Use Strategy 2** when: Fields follow naming patterns (internal_*, *_id, temp_*)
- **Use Strategy 3** when: Automating data quality cleanup (null %, cardinality)
- **Combine all 3** for: Maximum field reduction in multi-stage pipeline

**Performance insights:**
- Each strategy executes in 1-3 seconds on 1000 records
- Memory reduction scales linearly with field count
- Pattern matching adds minimal overhead vs. explicit lists

**Next steps:**
- Test with your own datasets
- Tune thresholds for optimal results
- Deploy to production environment
- Combine with other privacy operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)